## Bronze > Silver: Dedup, Enrichment Vehicles/Routes

In [0]:
# ---- Silver config  ----
CATALOG = "dbr_dev"
SCHEMA  = "live_transit_monitor"

BRONZE   = f"{CATALOG}.{SCHEMA}.gps_data"
VEHICLES = f"{CATALOG}.{SCHEMA}.bronze_vehicles"
ROUTES   = f"{CATALOG}.{SCHEMA}.bronze_gtfs_routes"
SILVER   = f"{CATALOG}.{SCHEMA}.gps_positions_silver"

In [0]:
# bronze.printSchema()

In [0]:
from pyspark.sql import functions as F

bronze = spark.read.table(BRONZE)

silver = (bronze
    .withColumn("generated", F.to_timestamp("generated"))
    .withColumn("lastUpdate", F.to_timestamp("lastUpdate"))
    .withColumn("scheduledTripStartTime", F.to_timestamp("scheduledTripStartTime"))
    .dropDuplicates(["vehicleId", "generated"])
    .withColumn("delay_min", F.round(F.col("delay") / 60.0, 1))
    .withColumn("has_trip", F.col("tripId").isNotNull())   # vehicle currently on a scheduled trip
    .withColumn("delay_bucket",
        F.when(F.col("delay") < -60, "early")
         .when(F.col("delay") <= 120, "on_time")
         .otherwise("delayed"))
    .withColumn("is_delayed", F.col("delay")>120)
    .withColumn("is_stopped", F.col("speed")==0)
    .withColumn("is_moving", F.col("speed") > 0)
    .withColumn("gps_ok",    F.col("gpsQuality") > 0)
    .filter(F.col("lat").between(54.2, 54.6) &    F.col("lon").between(18.3, 19.0)
)
)


In [0]:
# display(silver.head(10))

In [0]:
routes = spark.read.table(ROUTES)

# routes.printSchema()


In [0]:
# display(routes.limit(10))

In [0]:
routes_enrichment = (
    routes.select(
        F.col("route_id").alias("gtfs_route_id"),
        "route_type",
        "route_color",
        "route_text_color",

    )
    .dropDuplicates(["gtfs_route_id"])
)

# display(routes_enrichment.limit(10))

In [0]:
vehicles = spark.read.table(VEHICLES)

# display(vehicles.limit(10))

In [0]:
# Choosing the columns we want to keep
vehicles_enrichment = (
    vehicles.select(
        "vehicleCode",
        "transportationType",
        "vehicleCharacteristics",
        "brand",
        "model",
        "productionYear",
        "length",
        "seats",
        "standingPlaces",
        "floorHeight",
        "driveType",
        F.col("carrirer").alias("carrier"),
        "airConditioning",
        "wheelchairsRamp",
        "ticketMachine",
        "usb"
    )
    .dropDuplicates(["vehicleCode"])
)

In [0]:
silver_enriched = (
    silver.join(
        routes_enrichment, silver["routeId"] == routes_enrichment["gtfs_route_id"], "left"
    )
    .drop("gtfs_route_id")
    .join(
        vehicles_enrichment, on="vehicleCode",how="left")
)
# display(silver_enriched.limit(10))

In [0]:
# Checking if schema is correct
# silver_enriched.printSchema()


In [0]:
silver = silver_enriched

In [0]:
(silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.gps_positions_silver"))

In [0]:
# Sanity check
s = spark.read.table(SILVER)
print("rows:", s.count())
s.select("routeShortName","headsign","delay_bucket","transportationType","model").show(10, truncate=False)